# AI Platform Agent Demo
This notebook demonstrates a **Supervisor-Worker** topology using LangGraph, simulating the patterns required for the Production Agent Platform role.

In [ ]:
# Install dependencies
# !pip install langgraph langchain-openai langchain-anthropic

## 1. Define the State
We need a state that tracks the conversation history and which worker is currently active.

In [ ]:
import operator
from typing import Annotated, List, TypedDict, Union
from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(TypedDict):
    # Annotated with operator.add so that new messages are appended to the list
    messages: Annotated[List[BaseMessage], operator.add]
    next_node: str
    # Track if the research is complete
    research_complete: bool

## 2. Define the Nodes (Supervisor & Workers)
The Supervisor decides who should work next. The Researcher worker simulates a tool call (MCP style).

In [ ]:
def supervisor(state: AgentState):
    print("--- SUPERVISOR: Deciding next step ---")
    last_message = state['messages'][-1].content
    
    if "research" in last_message.lower() and not state.get('research_complete'):
        return {"next_node": "researcher"}
    else:
        return {"next_node": "end"}

def researcher_worker(state: AgentState):
    print("--- RESEARCHER: Performing lookup ---")
    # In a real app, this would be an MCP tool call
    return {
        "messages": [HumanMessage(content="Research found: The Model Context Protocol (MCP) is a new standard from Anthropic.")],
        "research_complete": True
    }

## 3. Construct the Graph

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("supervisor", supervisor)
workflow.add_node("researcher", researcher_worker)

# Add edges
workflow.set_entry_point("supervisor")

# Conditional edges from supervisor
workflow.add_conditional_edges(
    "supervisor",
    lambda x: x["next_node"],
    {
        "researcher": "researcher",
        "end": END
    }
)

# Always go back to supervisor after research
workflow.add_edge("researcher", "supervisor")

app = workflow.compile()

## 4. Run the Agent

In [ ]:
inputs = {"messages": [HumanMessage(content="Please do some research on MCP.")]}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Output from node '{key}':")
        # print(value)
    print("---")

## Key Takeaways for the Interview:
1. **State Isolation:** Notice how `AgentState` manages data flow.
2. **Cycle Management:** The edge from `researcher` back to `supervisor` creates a loop.
3. **Modular Design:** Each node is a pure function (or nearly so), making them easy to test in isolation.